In [36]:
from enum import Enum


class State(Enum):
    """DFA States for valid number recognition."""
    START = 0           # Initial state
    SIGN = 1            # After optional sign
    INTEGER = 2         # After one or more digits (accepting)
    DECIMAL = 3         # After optional dot (accepting if has digit)
    DOT = 4             # After dot without preceding digit (must see digit next)
    EXP = 5             # After exponent marker (e/E)
    EXP_SIGN = 6        # After optional sign in exponent
    EXP_INTEGER = 7     # After one or more digits in exponent (accepting)


class CharType(Enum):
    """Character type classifications for DFA transitions."""
    SIGN = 'sign'
    DIGIT = 'digit'
    DOT = 'dot'
    EXP = 'exp'
    INVALID = 'invalid'


class Solution:
    """
    Valid Number - Formal DFA Implementation
    
    Validates if a string represents a valid number using a Deterministic Finite Automaton.
    
    Rules enforced by transition table:
    1. Must start with sign, dot, or digit (only these transitions from START)
    2. Can only see digit before exp (no transistion to EXP from SIGN/DECIMAL in reverse)
    3. Can only visit exp once (EXP is one-way, no path back)
    4. Can only visit dot once (only INTEGER→DECIMAL transition adds dot, not bidirectional)
    5. Must visit digit at least once (only 3 accepting states involve digits)
    6. Must end with digit or dot (accepting states are INTEGER/DECIMAL/EXP_INTEGER only)
    
    Time Complexity: O(n) - single pass through string
    Space Complexity: O(1) - constant number of states
    """
    
    def __init__(self):
        # Character set classification
        self.digits = {str(i): True for i in range(10)}
        self.exps = {'e': True, 'E': True}
        self.signs = {'+': True, '-': True}
        self.dots = {'.': True}
        
        # Transition table: (current_state, char_type) → next_state
        # Any transition not in this table → invalid (rejection)
        self.transitions = {
            # From START: optional sign or digit or dot
            (State.START, CharType.SIGN): State.SIGN,
            (State.START, CharType.DIGIT): State.INTEGER,
            (State.START, CharType.DOT): State.DOT,
            
            # From SIGN: must see digit or dot
            (State.SIGN, CharType.DIGIT): State.INTEGER,
            (State.SIGN, CharType.DOT): State.DOT,
            
            # From DOT: must see digit
            (State.DOT, CharType.DIGIT): State.DECIMAL,

            # From INTEGER: can continue with digit, see dot (once), or see exp
            (State.INTEGER, CharType.DIGIT): State.INTEGER,
            (State.INTEGER, CharType.DOT): State.DECIMAL,
            (State.INTEGER, CharType.EXP): State.EXP,
            
            # From DECIMAL: can continue with digit or see exp
            (State.DECIMAL, CharType.DIGIT): State.DECIMAL,
            (State.DECIMAL, CharType.EXP): State.EXP,
            
            # From EXP: optional sign or must see digit
            (State.EXP, CharType.SIGN): State.EXP_SIGN,
            (State.EXP, CharType.DIGIT): State.EXP_INTEGER,
            
            # From EXP_SIGN: must see digit
            (State.EXP_SIGN, CharType.DIGIT): State.EXP_INTEGER,
            
            # From EXP_INTEGER: can continue with digit
            (State.EXP_INTEGER, CharType.DIGIT): State.EXP_INTEGER,
        }
        
        # Accepting states: valid ending positions (ensures digit visited)
        self.accepting_states = {State.INTEGER, State.DECIMAL, State.EXP_INTEGER}
    
    def _char_type(self, char: str) -> CharType:
        """Classify a character into a type for transition lookup."""
        if char in self.digits:
            return CharType.DIGIT
        elif char in self.dots:
            return CharType.DOT
        elif char in self.exps:
            return CharType.EXP
        elif char in self.signs:
            return CharType.SIGN
        else:
            return CharType.INVALID
    
    def isNumber(self, s: str) -> bool:
        """
        Validates if string s represents a valid number using DFA transitions.
        
        Args:
            s: String to validate
            
        Returns:
            True if s is a valid number, False otherwise
        """
        if not s:
            return False
        
        state = State.START
        
        # Process each character, following transition table
        for char in s:
            char_type = self._char_type(char)
            
            # Look up next state in transition table
            if (state, char_type) not in self.transitions:
                return False  # Invalid transition → reject
            
            state = self.transitions[(state, char_type)]
        
        # Accept only if final state is an accepting state
        return state in self.accepting_states


In [37]:
x = "-.E3"

sol = Solution()

y = sol.isNumber(x)

print(y)

False


In [38]:
# any char in x is a digit
